# Module 04 — PyTorch Tensors & Autograd

In Module 01 we hand-built a scalar autograd engine: the `Value` class,
with backward rules coded by hand for `+`, `*`, `**`, `tanh`, and `relu`.
Every gradient in that engine was one number flowing through one scalar
node at a time.

This module redoes the *same idea* — build an expression out of
differentiable operations, then automatically compute gradients by walking
the computation graph backward — using real `torch.Tensor` objects and
`.backward()` instead. The concept doesn't change; what changes is that
PyTorch's autograd is vectorized (works on whole arrays at once, not one
scalar at a time) and GPU-ready (the exact same code can run on a CUDA
device). No neuron/layer/training-loop structure here — that's Module 05
(`nn.Module` & optimizers); this module is just tensors + autograd.

## 1. A scalar example, mirroring Module 01

In [ ]:
import torch

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Just like Module 01's `Value(2.0)`, a `torch.Tensor` becomes a node in a
computation graph the moment we ask PyTorch to track gradients for it via
`requires_grad=True`. We build the same kind of small expression as Module
01 did by hand, then let `.backward()` do the graph traversal instead of
our own hand-written `_backward` closures.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-3.0, requires_grad=True)
c = torch.tensor(10.0, requires_grad=True)

e = a * b            # e = -6
d = e + c            # d = 4
f = torch.tensor(-2.0, requires_grad=True)
L = d * f            # L = -8

L.backward()

print("L =", L.item())
print("dL/da =", a.grad.item())
print("dL/db =", b.grad.item())
print("dL/dc =", c.grad.item())
print("dL/df =", f.grad.item())

**Verify against hand-computed derivatives**, the same sanity check
Module 01 used: `L = (a*b + c) * f`, so
`dL/da = b*f`, `dL/db = a*f`, `dL/dc = f`, `dL/df = a*b + c`.

In [ ]:
expected_da = (b * f).item()
expected_db = (a * f).item()
expected_dc = f.item()
expected_df = (a * b + c).item()

assert abs(a.grad.item() - expected_da) < 1e-6
assert abs(b.grad.item() - expected_db) < 1e-6
assert abs(c.grad.item() - expected_dc) < 1e-6
assert abs(f.grad.item() - expected_df) < 1e-6
print("All scalar gradients match the hand-computed derivatives.")

## 2. The same graph, but vectorized

This is the part Module 01's scalar `Value` engine couldn't do. Instead of
one number per node, each node now holds a whole tensor — the *same*
`+`, `*`, `tanh` operations broadcast across every element, and one call to
`.backward()` computes every gradient in the batch simultaneously. This is
exactly why frameworks vectorize: a neural net layer processing 1000
examples does 1000x the work with the same one line of code and one
backward pass, instead of a Python loop over 1000 scalar graphs.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, -1.0], requires_grad=True)
w = torch.tensor([0.5, -0.5, 1.0, 2.0], requires_grad=True)

z = x * w                 # elementwise product
y = torch.tanh(z).sum()   # squash each element, then reduce to a scalar loss

y.backward()

print("z =", z.detach())
print("y =", y.item())
print("dy/dx =", x.grad)
print("dy/dw =", w.grad)

**Verify against the hand-computed derivative.** For
`y = sum(tanh(x_i * w_i))`, the chain rule gives
`dy/dx_i = w_i * (1 - tanh(x_i*w_i)^2)` and symmetrically for `dy/dw_i`.
We recompute that by hand (using plain tensor math, not autograd) and
compare against what `.backward()` produced above.

In [ ]:
z_vals = (x * w).detach()
sech2 = 1 - torch.tanh(z_vals) ** 2   # d/dz tanh(z) = 1 - tanh(z)^2

expected_dx = w.detach() * sech2
expected_dw = x.detach() * sech2

assert torch.allclose(x.grad, expected_dx, atol=1e-6)
assert torch.allclose(w.grad, expected_dw, atol=1e-6)
print("Vectorized gradients match the hand-computed derivatives elementwise.")

## 3. Gradients accumulate — `grad_fn` and `zero_()`

Two things about PyTorch's autograd that trip people up the first time:

1. Only *leaf* tensors with `requires_grad=True` keep a `.grad` — intermediate
   tensors like `z` above only expose their `grad_fn` (the operation that
   produced them), not a `.grad`, unless you explicitly ask to retain it.
2. Calling `.backward()` again **adds** to `.grad` instead of replacing it —
   this is what lets gradient accumulation over multiple mini-batches work,
   but it means you must zero gradients yourself between independent
   backward passes (Module 05's optimizers do this via `optimizer.zero_grad()`).

In [ ]:
print("z.grad_fn:", z.grad_fn)       # an intermediate node: has grad_fn, not .grad
print("x.grad_fn:", x.grad_fn)       # a leaf: no grad_fn, it IS the graph input

# Demonstrate accumulation: running backward again without zeroing adds on top
y2 = torch.tanh(x * w).sum()
y2.backward()
print("x.grad after a second backward (accumulated):", x.grad)

x.grad.zero_()
w.grad.zero_()
y3 = torch.tanh(x * w).sum()
y3.backward()
print("x.grad after zero_() + a fresh backward:", x.grad)
assert torch.allclose(x.grad, expected_dx, atol=1e-6)
print("Matches the single-pass gradient again, as expected.")

## 4. GPU-readiness

The entire point of switching to `torch.Tensor` is that the exact same code
above runs on a GPU with no changes beyond moving the tensors there via
`.to(device)`. This cell is written to work whether or not this machine has
a CUDA GPU — on Colab with a GPU runtime, `device` below will be `"cuda"`
and the computation actually happens on the GPU.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

x_dev = torch.tensor([1.0, 2.0, 3.0, -1.0], requires_grad=True, device=device)
w_dev = torch.tensor([0.5, -0.5, 1.0, 2.0], requires_grad=True, device=device)

y_dev = torch.tanh(x_dev * w_dev).sum()
y_dev.backward()

print("y (on", device, "):", y_dev.item())
print("x.grad (on", device, "):", x_dev.grad)

## Recap

- Module 01's hand-written `_backward` closures and Module 04's
  `.backward()` are doing the *same conceptual thing*: walking a
  computation graph in reverse and applying the chain rule at each node.
- The difference is entirely about scale and hardware: real tensors let one
  graph node hold a whole array, and the same code moves to a GPU by
  changing which `device` a tensor lives on.
- Next up — Module 05: `nn.Module` and optimizers, redoing the
  `Neuron`/`Layer`/`MLP` structure and the training loop from Modules 02-03,
  but with PyTorch's built-in building blocks instead of hand-rolled ones.